In [1]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/final.csv")


In [ ]:
print(df.columns)

Index(['time_id', 'seconds_in_bucket', '0', '1', '2', '3', '4', '5', '6', '7',
       ...
       '115', '116', '118', '119', '120', '122', '123', '124', '125', '126'],
      dtype='object', length=114)


In [ ]:
print(df.head())

NameError: name 'df' is not defined

In [ ]:
"""
tcn_cluster_pipeline.py — Multi-Stock Cluster TCN Volatility
==============================================================

Architecture (480 raw seconds input):
  • Input         : (B, K, 480, 2) — 480 time steps, 2 feature channels
  • TCN           : 8 stacked residual blocks, kernel_size=3,
                     dilations = [1,2,4,8,16,32,64,128], hidden_dim=64
  • Head          : Global avg pool over time → Linear(64, 1) → (B, K)

Windowing
---------
  Each time_id has exactly 600 raw seconds.
  One non-overlapping window:
      input  : raw secs [  0 : 480)
      target : raw secs [480 : 600)  → 120 raw seconds for future RV

  SAMPLES_PER_TID = 1

2 Feature Channels (computed on raw seconds)
---------------------------------------------
  0  return          — clipped log-return
  1  log_wap         — log(WAP / open)

Evaluation
----------
  Both smearing-corrected and raw (non-smearing) RV metrics are reported.
"""

import warnings
import time as _time

import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
try:
    from kneed import KneeLocator
    _KNEED_AVAILABLE = True
except ImportError:
    _KNEED_AVAILABLE = False


# ─────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────
TOTAL_LEN   = 600           # raw seconds per time_id
INPUT_LEN   = 480           # raw seconds consumed by input
TARGET_LEN  = 120           # raw seconds for future RV
SAMPLES_PER_TID = 1         # one window fills the whole block

EPSILON    = 1e-5
RET_CLIP   = 0.05

CHANNEL_NAMES = ["return", "log_wap"]
N_CHANNELS   = len(CHANNEL_NAMES)
PROFILE_COLS = ["rv", "mean_ret", "skew", "kurt", "max_dd"]


# ─────────────────────────────────────────────────────────────────────
# Low‑level feature helpers (unchanged)
# ─────────────────────────────────────────────────────────────────────
def _safe_log_return_matrix(mat: np.ndarray) -> np.ndarray:
    safe = np.where(mat > 0, mat, 1e-6).astype(np.float32)
    ret = np.zeros_like(safe)
    ret[1:] = np.log(safe[1:] / safe[:-1])
    np.clip(ret, -RET_CLIP, RET_CLIP, out=ret)
    np.nan_to_num(ret, copy=False, nan=0.0, posinf=RET_CLIP, neginf=-RET_CLIP)
    return ret

def robust_fill(mat: np.ndarray) -> np.ndarray:
    bad = ~np.isfinite(mat) | (mat <= 0)
    if not bad.any():
        return mat
    out = mat.copy()
    rows = out.shape[0]
    row_idx = np.arange(rows)[:, None]
    fwd = np.where(~bad, row_idx, -1)
    np.maximum.accumulate(fwd, axis=0, out=fwd)
    rev = np.where(~bad, row_idx, rows)
    bwd = rows - 1 - np.maximum.accumulate(
        (rows - 1 - rev)[::-1], axis=0
    )[::-1]
    use = np.where(fwd >= 0, fwd, bwd)
    all_bad = (use < 0) | (use >= rows)
    use = np.where(all_bad, 0, use)
    out = out[use, np.arange(out.shape[1])]
    return np.where(all_bad, 1.0, out)

def compute_features_matrix(wap_mat: np.ndarray,
                             open_prices: np.ndarray) -> np.ndarray:
    ret = _safe_log_return_matrix(wap_mat)
    safe_wap = np.maximum(wap_mat, 1e-6).astype(np.float32)
    log_wap = np.log(safe_wap / (open_prices[None, :] + EPSILON)).astype(np.float32)
    feat = np.stack([ret, log_wap], axis=-1)
    np.nan_to_num(feat, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return feat


# ─────────────────────────────────────────────────────────────────────
# Split, profiling, clustering (unchanged)
# ─────────────────────────────────────────────────────────────────────
def make_splits_from_time_id(df, val_ratio=0.15, test_ratio=0.15, seed=42):
    rng = np.random.default_rng(seed)
    tids = df["time_id"].unique().copy()
    rng.shuffle(tids)
    n = len(tids)
    n_test = int(n * test_ratio)
    n_val = int(n * val_ratio)
    return {"split_summary": {
        "train": {"ids": tids[n_test + n_val:]},
        "val":   {"ids": tids[n_test : n_test + n_val]},
        "test":  {"ids": tids[:n_test]},
    }}

def _vectorised_window_stats(ret_mat: np.ndarray) -> np.ndarray:
    n_stocks = ret_mat.shape[1]
    out = np.zeros((n_stocks, 5), dtype=np.float32)
    out[:, 0] = np.sqrt(np.einsum("ij,ij->j", ret_mat, ret_mat))
    out[:, 1] = ret_mat.mean(axis=0)
    std_col = ret_mat.std(axis=0)
    nontrivial = std_col > 1e-12
    if nontrivial.any():
        out[nontrivial, 2] = skew(ret_mat[:, nontrivial], axis=0, bias=False)
        out[nontrivial, 3] = kurtosis(ret_mat[:, nontrivial], axis=0,
                                      bias=False, fisher=True)
    cum_ret = np.exp(np.cumsum(ret_mat, axis=0))
    running_max = np.maximum.accumulate(cum_ret, axis=0)
    out[:, 4] = ((running_max - cum_ret) / (running_max + EPSILON)).max(axis=0)
    np.nan_to_num(out, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return out

def build_stock_profiles(df, stock_cols, train_ids):
    print("Step 1 — Building stock profiles (train only, raw input) …")
    df_s = df[df["time_id"].isin(train_ids)].sort_values(["time_id", "seconds_in_bucket"])
    stat_sum, stat_count = np.zeros((len(stock_cols), len(PROFILE_COLS)), dtype=np.float64), 0
    for _, grp in df_s.groupby("time_id", sort=False):
        mat = robust_fill(grp[stock_cols].values.astype(np.float32))
        if mat.shape[0] < TOTAL_LEN:
            continue
        # use input window only for profile (480s)
        inp_raw = mat[:INPUT_LEN]
        ret_raw = _safe_log_return_matrix(inp_raw)
        stat_sum += _vectorised_window_stats(ret_raw).astype(np.float64)
        stat_count += 1
    profiles = pd.DataFrame((stat_sum / max(stat_count, 1)).astype(np.float32),
                            index=stock_cols, columns=PROFILE_COLS)
    print(f"  Profiles: {profiles.shape}  ({stat_count} train time_ids)")
    return profiles

def find_optimal_k_elbow(profiles, k_range=range(2, 21), random_state=42):
    X = StandardScaler().fit_transform(profiles.values)
    inertias = [KMeans(k, n_init=10, max_iter=300, random_state=random_state).fit(X).inertia_ for k in k_range]
    if _KNEED_AVAILABLE:
        kl = KneeLocator(list(k_range), inertias, curve="convex", direction="decreasing")
        return kl.elbow if kl.elbow else 10
    return 10

def cluster_stocks_kmeans(profiles, n_clusters=10, seed=42):
    print(f"\nStep 2 — K-Means (k={n_clusters}) …")
    X = StandardScaler().fit_transform(profiles.values)
    km = KMeans(n_clusters=n_clusters, n_init=20, max_iter=500, random_state=seed)
    labels = km.fit_predict(X)
    cm = pd.Series(labels, index=profiles.index, name="cluster_id")
    print(f"  Cluster sizes: {cm.value_counts().sort_index().to_dict()}")
    return cm


# ─────────────────────────────────────────────────────────────────────
# TCN MODULE  (residual blocks with dilated causal conv)
# ─────────────────────────────────────────────────────────────────────
class CausalConv1dBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()
        self.conv = nn.Conv1d(
            in_channels, out_channels, kernel_size,
            dilation=dilation,
            padding=(kernel_size - 1) * dilation   # causal padding
        )
        self.weight_norm = nn.utils.weight_norm(self.conv)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) \
                          if in_channels != out_channels else None

    def forward(self, x):
        # x: (B, C, T)
        out = self.conv(x)
        # remove trailing padded values to keep causal
        out = out[..., :-self.conv.dilation[0] * (self.conv.kernel_size[0] - 1)]
        out = self.relu(out)
        out = self.dropout(out)
        if self.downsample is not None:
            x = self.downsample(x)
        return out + x


class TCNEncoder(nn.Module):
    """Multi‑scale TCN that processes (B, C_in, T) → (B, hidden_dim)"""
    def __init__(self,
                 input_dim=2,
                 hidden_dim=64,
                 dilations=(1, 2, 4, 8, 16, 32, 64, 128),
                 kernel_size=3,
                 dropout=0.2):
        super().__init__()
        layers = []
        in_channels = input_dim
        for d in dilations:
            layers.append(
                CausalConv1dBlock(in_channels, hidden_dim, kernel_size, d, dropout)
            )
            in_channels = hidden_dim   # after first layer, channels stay fixed
        self.network = nn.Sequential(*layers)
        self.hidden_dim = hidden_dim

    def forward(self, x):
        # x: (B, T, C) → permute to (B, C, T)
        x = x.permute(0, 2, 1)
        out = self.network(x)          # (B, hidden_dim, T')
        # Global average pool over time
        out = out.mean(dim=2)          # (B, hidden_dim)
        return out


# ─────────────────────────────────────────────────────────────────────
# DATASET  (480s raw input, 120s raw target)
# ─────────────────────────────────────────────────────────────────────
class ClusterTCNDataset(Dataset):
    def __init__(self, df, stock_cols, cluster_map, time_ids, name=""):
        self.stock_cols = stock_cols
        cluster_ids = sorted(cluster_map.unique())
        self.cluster_ids = cluster_ids
        self.n_clusters = len(cluster_ids)
        self.cluster_col_idx = {
            cid: np.array([stock_cols.index(s) for s in cluster_map.index[cluster_map == cid]],
                          dtype=np.int32)
            for cid in cluster_ids
        }

        df = df[df["time_id"].isin(time_ids)].sort_values(["time_id", "seconds_in_bucket"])
        self._samples = []
        self._raw_blocks = {}
        for tid, grp in df.groupby("time_id", sort=True):
            mat = robust_fill(grp[stock_cols].values.astype(np.float32))
            if mat.shape[0] < TOTAL_LEN:
                continue
            self._raw_blocks[tid] = mat[:TOTAL_LEN]      # (600, S)
            self._samples.append(tid)

        n_tids = len(self._raw_blocks)
        mb = sum(v.nbytes for v in self._raw_blocks.values()) / 1e6
        print(f"  [{name:5s}] {n_tids:>5} time_ids × 1 window "
              f"= {len(self._samples)} samples | cache {mb:.1f} MB")

    def __len__(self):
        return len(self._samples)

    def __getitem__(self, idx):
        tid = self._samples[idx]
        mat = self._raw_blocks[tid]       # (600, S)

        raw_inp = mat[:INPUT_LEN]         # (480, S)
        raw_tgt = mat[INPUT_LEN:INPUT_LEN+TARGET_LEN]   # (120, S)
        open_prices = raw_inp[0]          # (S,)

        # Features on raw input
        feat_all = compute_features_matrix(raw_inp, open_prices)   # (480, S, 2)
        ret_inp = _safe_log_return_matrix(raw_inp)                # (480, S)
        ret_tgt = _safe_log_return_matrix(raw_tgt)                # (120, S)

        x_seq     = np.empty((self.n_clusters, INPUT_LEN, N_CHANNELS), dtype=np.float32)
        targets   = np.empty(self.n_clusters, dtype=np.float32)
        rv_inputs = np.empty(self.n_clusters, dtype=np.float32)

        for ci, cid in enumerate(self.cluster_ids):
            idx_c = self.cluster_col_idx[cid]
            # mean feature across stocks in cluster
            x_seq[ci] = feat_all[:, idx_c, :].mean(axis=1)   # (480, 2)

            # RV_input from 480s raw returns
            r_in = ret_inp[:, idx_c]
            rv_in = float(np.sqrt(np.einsum("ij,ij->j", r_in, r_in)).mean()) + EPSILON
            rv_inputs[ci] = rv_in

            # RV_future from 120s raw returns
            r_fut = ret_tgt[:, idx_c]
            rv_fut = float(np.sqrt(np.einsum("ij,ij->j", r_fut, r_fut)).mean()) + EPSILON
            targets[ci] = np.log(rv_fut / rv_in)

        np.nan_to_num(x_seq, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        return (torch.from_numpy(x_seq),     # (K, 480, 2)
                torch.from_numpy(targets),   # (K,)
                torch.from_numpy(rv_inputs)) # (K,)


# ─────────────────────────────────────────────────────────────────────
# MODEL  (TCN → MLP)
# ─────────────────────────────────────────────────────────────────────
class ClusterVolatilityTCN(nn.Module):
    """
    Multi‑stock Cluster TCN.
    Input  : (B, K, 480, 2)
    TCN    : shared across clusters, outputs (B*K, hidden_dim)
    Head   : Linear(hidden_dim, 1) → (B, K)
    """
    def __init__(self, n_clusters=10, input_dim=N_CHANNELS,
                 hidden_dim=64, dropout=0.2):
        super().__init__()
        self.n_clusters = n_clusters
        self.tcn = TCNEncoder(input_dim=input_dim,
                              hidden_dim=hidden_dim,
                              dropout=dropout)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x_seq):
        B, K, T, F = x_seq.shape
        x_flat = x_seq.reshape(B * K, T, F)      # (B*K, 480, 2)
        z = self.tcn(x_flat)                      # (B*K, hidden_dim)
        out = self.head(z).squeeze(1)             # (B*K,)
        return out.reshape(B, K)


# ─────────────────────────────────────────────────────────────────────
# Training, smearing, evaluation (unchanged except names)
# ─────────────────────────────────────────────────────────────────────
class QLIKELoss(nn.Module):
    def forward(self, y_pred, y_true):
        eps = y_true - y_pred
        return (torch.exp(eps) - eps - 1.0).mean()

def get_loss_fn(loss_type="qlike"):
    return QLIKELoss() if loss_type == "qlike" else nn.MSELoss()

def train_model(model, train_loader, val_loader, epochs=50, lr=1e-3,
                patience=10, loss_type="qlike", ckpt_path="best_cluster_tcn.pt"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=5)
    loss_fn = get_loss_fn(loss_type)
    print(f"  Loss: {loss_type.upper()} | Device: {device}")

    best, stale = float("inf"), 0
    for epoch in range(1, epochs + 1):
        t0 = _time.time()
        model.train()
        tr, n = 0.0, 0
        for x_seq, yb, _ in train_loader:
            x_seq, yb = x_seq.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(x_seq), yb)
            if not torch.isfinite(loss):
                continue
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            tr += loss.item() * x_seq.size(0)
            n += x_seq.size(0)
        tr /= max(n, 1)

        model.eval()
        vl, n = 0.0, 0
        with torch.no_grad():
            for x_seq, yb, _ in val_loader:
                x_seq, yb = x_seq.to(device), yb.to(device)
                vl += loss_fn(model(x_seq), yb).item() * x_seq.size(0)
                n += x_seq.size(0)
        vl /= max(n, 1)

        sched.step(vl)
        improved = vl < best
        tag = ""
        if improved:
            best, stale = vl, 0
            torch.save(model.state_dict(), ckpt_path)
            tag = " ★"
        else:
            stale += 1

        if epoch == 1 or epoch % 5 == 0 or improved:
            print(f"  Epoch {epoch:>3}/{epochs} | Train {tr:.6f} | Val {vl:.6f} | "
                  f"{_time.time()-t0:.1f}s{tag}")
        if stale >= patience:
            print(f"  Early stop at epoch {epoch}")
            break

    model.load_state_dict(torch.load(ckpt_path, map_location="cpu"))
    return model

def compute_smearing_factor(model, loader):
    device = next(model.parameters()).device
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for x_seq, yb, _ in loader:
            preds.append(model(x_seq.to(device)).cpu().numpy())
            actuals.append(yb.numpy())
    preds = np.concatenate(preds)
    actuals = np.concatenate(actuals)
    delta = np.mean(np.exp(actuals - preds), axis=0)
    print(f"  Per-cluster smearing δ̂: {np.round(delta, 4).tolist()}")
    return delta

def _rv_metrics(rv_pred, rv_act):
    fp = np.maximum(rv_pred.ravel(), EPSILON)
    fa = np.maximum(rv_act.ravel(), EPSILON)
    mse = mean_squared_error(fa, fp)
    pct = (fp - fa) / fa
    ratio = fa / (fp + EPSILON)
    return dict(
        mse=float(mse), rmse=float(np.sqrt(mse)),
        r2=float(r2_score(fa, fp)), mae=float(np.mean(np.abs(fp - fa))),
        rmspe=float(np.sqrt(np.mean(pct**2))),
        mape=float(np.mean(np.abs(pct))),
        qlike=float(np.mean(ratio - np.log(ratio + EPSILON) - 1))
    )

def _print_metrics(m, label):
    w = 48
    print(f"\n  ┌{'─'*w}┐")
    print(f"  │  {label:<{w-2}}│")
    print(f"  ├{'─'*w}┤")
    print(f"  │  MSE   : {m['mse']:<12.8f}   RMSE  : {m['rmse']:<12.8f}│")
    print(f"  │  R²    : {m['r2']:<12.6f}   MAE   : {m['mae']:<12.8f}│")
    print(f"  │  RMSPE : {m['rmspe']:<12.6f}   MAPE  : {m['mape']:<12.6f}│")
    print(f"  │  QLIKE : {m['qlike']:<12.6f}{'':>22}│")
    print(f"  └{'─'*w}┘")

def evaluate_model(model, loader, smearing, label=""):
    device = next(model.parameters()).device
    model.eval()
    logp_list, logy_list, rvin_list = [], [], []
    with torch.no_grad():
        for x_seq, yb, rv_in in loader:
            logp_list.append(model(x_seq.to(device)).cpu().numpy())
            logy_list.append(yb.numpy())
            rvin_list.append(rv_in.numpy())
    logp = np.concatenate(logp_list)
    logy = np.concatenate(logy_list)
    rvin = np.concatenate(rvin_list)
    rv_act = np.exp(logy) * rvin
    rv_raw = np.exp(logp) * rvin
    m_raw = _rv_metrics(rv_raw, rv_act)
    rv_sme = np.exp(logp) * smearing[None, :] * rvin
    m_sme = _rv_metrics(rv_sme, rv_act)
    if label:
        print(f"\n{'═'*52}\n  {label}\n{'═'*52}")
    _print_metrics(m_raw, "No Smearing   — exp(ŷ) × RV_input")
    _print_metrics(m_sme, "Smearing-Corrected — exp(ŷ) × δ̂ × RV_input")
    return {"no_smearing": m_raw, "smearing": m_sme}


# ─────────────────────────────────────────────────────────────────────
# FULL PIPELINE
# ─────────────────────────────────────────────────────────────────────
def run_tcn_cluster_pipeline(
    df, stock_cols, splits,
    n_clusters=None, k_range=range(2, 21), seed=42,
    hidden_dim=64, dropout=0.2,
    epochs=50, bs=32, lr=1e-3, patience=10,
    loss_type="qlike", workers=0,
    ckpt_path="best_cluster_tcn.pt"
):
    print("="*60)
    print("Multi-Stock Cluster TCN Volatility Pipeline")
    print(f"  Windowing : {INPUT_LEN}s input + {TARGET_LEN}s target = 600s total")
    print(f"  Input shape : (K, {INPUT_LEN}, {N_CHANNELS})")
    print(f"  TCN : dilations=[1,2,4,8,16,32,64,128], hidden_dim={hidden_dim}")
    print("="*60)

    train_ids = splits["split_summary"]["train"]["ids"]
    val_ids   = splits["split_summary"]["val"]["ids"]
    test_ids  = splits["split_summary"]["test"]["ids"]
    print(f"Split: train={len(train_ids)} | val={len(val_ids)} | test={len(test_ids)}")

    profiles = build_stock_profiles(df, stock_cols, train_ids)
    if n_clusters is None:
        n_clusters = find_optimal_k_elbow(profiles, k_range=k_range, random_state=seed)
        print(f"  Elbow → k={n_clusters}")
    cluster_map = cluster_stocks_kmeans(profiles, n_clusters, seed)

    print(f"\nBuilding datasets ({INPUT_LEN}s raw input, {TARGET_LEN}s raw target) …")
    train_ds = ClusterTCNDataset(df, stock_cols, cluster_map, train_ids, "train")
    val_ds   = ClusterTCNDataset(df, stock_cols, cluster_map, val_ids, "val")
    test_ds  = ClusterTCNDataset(df, stock_cols, cluster_map, test_ids, "test")

    pin = torch.cuda.is_available()
    tl = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=workers, pin_memory=pin)
    vl = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=workers, pin_memory=pin)
    sl = DataLoader(test_ds, batch_size=bs, shuffle=False, num_workers=workers, pin_memory=pin)

    x_seq, yb, rv_in = next(iter(tl))
    print(f"\nSanity: seq {tuple(x_seq.shape)} | y {tuple(yb.shape)} | rv_in {tuple(rv_in.shape)}")
    assert x_seq.shape[2] == INPUT_LEN and x_seq.shape[3] == N_CHANNELS
    assert torch.isfinite(x_seq).all() and torch.isfinite(yb).all()

    model = ClusterVolatilityTCN(
        n_clusters=n_clusters, input_dim=N_CHANNELS,
        hidden_dim=hidden_dim, dropout=dropout
    )
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nModel: ClusterVolatilityTCN")
    print(f"  Input : (K={n_clusters}, T={INPUT_LEN}, F={N_CHANNELS})")
    print(f"  TCN   : hidden_dim={hidden_dim}, dropout={dropout}")
    print(f"  Head  : Linear({hidden_dim}, 1)")
    print(f"  Params: {n_params:,}\n")

    print("Training …")
    model = train_model(model, tl, vl, epochs=epochs, lr=lr, patience=patience,
                        loss_type=loss_type, ckpt_path=ckpt_path)

    print("\nSmearing factors (estimated on train set):")
    delta = compute_smearing_factor(model, tl)

    val_metrics = evaluate_model(model, vl, delta, "Validation — smearing vs raw")
    test_metrics = evaluate_model(model, sl, delta, "TEST SET — smearing vs raw [touched once]")

    return {
        "model": model, "profiles": profiles, "cluster_map": cluster_map,
        "smearing": delta, "val_metrics": val_metrics,
        "test_metrics": test_metrics, "n_clusters": n_clusters
    }


# ─────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────
def main(
    df, n_clusters=None, epochs=100, bs=32, lr=1e-3, patience=10,
    loss_type="mse", hidden_dim=64, dropout=0.2,
    seed=42, val_ratio=0.15, test_ratio=0.15, workers=0,
    ckpt_path="best_cluster_tcn.pt"
):
    print("Loading data …")
    stock_cols = [c for c in df.columns if c not in ("time_id", "seconds_in_bucket")]
    print(f"  shape: {df.shape} | stocks: {len(stock_cols)} | tids: {df['time_id'].nunique()}")
    splits = make_splits_from_time_id(df, val_ratio, test_ratio, seed)
    results = run_tcn_cluster_pipeline(
        df, stock_cols, splits,
        n_clusters=n_clusters, seed=seed, hidden_dim=hidden_dim,
        dropout=dropout, epochs=epochs, bs=bs, lr=lr, patience=patience,
        loss_type=loss_type, workers=workers, ckpt_path=ckpt_path
    )

    tm_raw = results["test_metrics"]["no_smearing"]
    tm_sme = results["test_metrics"]["smearing"]
    print("\n" + "="*60)
    print("Pipeline complete — Test Set Summary")
    print("="*60)
    print(f"  {'Metric':<10}  {'No Smearing':>14}  {'Smearing':>14}")
    print(f"  {'-'*10}  {'-'*14}  {'-'*14}")
    for key in ("qlike", "rmse", "rmspe", "mape", "r2"):
        print(f"  {key.upper():<10}  {tm_raw[key]:>14.6f}  {tm_sme[key]:>14.6f}")
    print("="*60)
    return results


if __name__ == "__main__":
    main(df)

Loading data …
  shape: (2298000, 114) | stocks: 112 | tids: 3830
Multi-Stock Cluster TCN Volatility Pipeline
  Windowing : 480s input + 120s target = 600s total
  Input shape : (K, 480, 2)
  TCN : dilations=[1,2,4,8,16,32,64,128], hidden_dim=64
Split: train=2682 | val=574 | test=574
Step 1 — Building stock profiles (train only, raw input) …
  Profiles: (112, 5)  (2682 train time_ids)
  Elbow → k=9

Step 2 — K-Means (k=9) …
  Cluster sizes: {0: 14, 1: 19, 2: 4, 3: 26, 4: 16, 5: 1, 6: 17, 7: 4, 8: 11}

Building datasets (480s raw input, 120s raw target) …
  [train]  2682 time_ids × 1 window = 2682 samples | cache 720.9 MB
  [val  ]   574 time_ids × 1 window = 574 samples | cache 154.3 MB
  [test ]   574 time_ids × 1 window = 574 samples | cache 154.3 MB

Sanity: seq (32, 9, 480, 2) | y (32, 9) | rv_in (32, 9)

Model: ClusterVolatilityTCN
  Input : (K=9, T=480, F=2)
  TCN   : hidden_dim=64, dropout=0.2
  Head  : Linear(64, 1)
  Params: 87,681

Training …
  Loss: QLIKE | Device: cpu
  Epo